# Qwen3-ASR Transcription

A lightweight pipeline to transcribe audio files to text using [Qwen3-ASR](https://huggingface.co/Qwen/Qwen3-ASR-0.6B).

**Steps:** Install deps → Load model → Upload audio → Transcribe → Save output

## Block 1 — Install dependencies

In [ ]:
# Install the official ASR Python package that supports model loading.
!pip install -q qwen-asr

# Install audio processing and torch
!pip install -q torch torchaudio librosa soundfile

## Block 2 — Import modules and check GPU

In [ ]:
import torch
import librosa
import soundfile as sf

from qwen_asr import Qwen3ASRModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Block 3 — Load the Qwen3-ASR model

Choose `Qwen/Qwen3-ASR-0.6B` (faster) or `Qwen/Qwen3-ASR-1.7B` (higher accuracy).

In [ ]:
model_name = "Qwen/Qwen3-ASR-0.6B"

asr_model = Qwen3ASRModel.from_pretrained(
    model_name,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

print("ASR model loaded:", model_name)

## Block 4 — Upload and play the audio file

In [ ]:
from google.colab import files
from IPython.display import Audio, display

uploaded = files.upload()
audio_file = list(uploaded.keys())[0]

print("Uploaded audio:", audio_file)

display(Audio(audio_file))

## Block 5 — Convert to mono 16kHz

In [ ]:
audio, sr = librosa.load(audio_file, sr=16000, mono=True)

clean_path = "clean_audio.wav"
sf.write(clean_path, audio, 16000)

print("Normalized audio saved as:", clean_path)

## Block 6 — Transcribe the audio

In [ ]:
results = asr_model.transcribe(
    audio=clean_path,
    language=None,       # Automatic language ID if None
    return_time_stamps=False
)

transcript = results[0].text

print("Transcription output:")
print(transcript)

## Block 7 — Save reference text

In [ ]:
with open("reference_text.txt", "w") as f:
    f.write(transcript)

print("Reference text saved to reference_text.txt")